In [128]:
import pandas as pd
import datetime
import sys
sys.path.append('\\\\htisec.local\\hk data\\Department\\PB\\PB\\001_Index_Arbitrage')
import cvxpy as cp

from Framework.pythonTool import *
from Framework.riskFactor import LogSizeFactor, IndustryFactor, RiskFactor
from typing import Dict, List, Optional, Union, Tuple
from Framework.dataTool import BarraDataLoader, IndexDataLoader, PriceDataLoader, PortfolioLoader
from Framework.riskTool import PortfolioExposureCalculator
from dataclasses import dataclass
from Framework.pythonTool import *

In [129]:
dol = ddb('bctmate')
date = '2026.01.26'

# barra相关数据获取工具
barra_data_loader = BarraDataLoader('2026-01-05')

# 指数权重获取工具
index_data_loader = IndexDataLoader('2026-01-05')

# 市场行情获取工具
price_data_loader = PriceDataLoader('2026-01-05')



In [130]:
# 从bctmate投资组合获取工具
portfolio_loader =  PortfolioLoader(date)

main_contracts = ['ZZ10_LS_SWAP001_L_01', 'ZZ10_LS_SWAP001_S_01', 'ZZ13_LS_SWAP001_L_01', 'ZZ13_LS_SWAP001_S_01']
portfolio = portfolio_loader.load_portfolio_from_bctmate(main_contracts=main_contracts, portfolio_name='MyPortfolio', replace_future=True)
portfolio_index = portfolio.loc[portfolio['asset_type'] == 'index']

In [131]:
portfolio_index

,code,quantity,close,market_value,asset_type,exchange_code
1804,000852.XSHG,-46800.0,8365.4345,-3.915023e+08,index,CCFX
1805,000905.XSHG,-40200.0,8506.6932,-3.419691e+08,index,CCFX
1806,000300.XSHG,-108600.0,4706.9640,-5.111763e+08,index,CCFX


In [170]:
# 根据以'上指数数量组合优化后，读取组合优化后标的
files = ['CSI1000 IMS交易指令模板.xlsx']
main_contracts = ['ZZ13_LS_SWAP001_L_01']

In [163]:
# 根据以'上指数数量组合优化后，读取组合优化后标的
files = ['CSI300 IMS交易指令模板.xlsx', 'CSI500 IMS交易指令模板.xlsx']
main_contracts = ['ZZ10_LS_SWAP001_L_01']

In [171]:
target_portfolio = []
for file in files:
    df = pd.read_excel(f'./Output/20260105/策略目标仓位/{file}', dtype={'代码': str})
    target_portfolio.append(df)
target_portfolio = pd.concat(target_portfolio)

target_portfolio['标的代码'] = target_portfolio['代码'].apply(lambda x: x + ' CH Equity')
target_portfolio = target_portfolio[['标的代码', '委托数量']]


# 从bctmate投资组合获取工具
portfolio_loader =  PortfolioLoader(date)
portfolio = portfolio_loader.load_portfolio_from_bctmate(main_contracts=main_contracts, portfolio_name='CSI1000', replace_future=True)

portfolio['标的代码'] = portfolio['code'].apply(lambda x: x.split('.')[0] + ' CH Equity')
current_portfolio = portfolio[['标的代码', 'quantity']]

target_portfolio = target_portfolio.set_index('标的代码')
current_portfolio = current_portfolio.set_index('标的代码')

merge_df = pd.merge(left=target_portfolio, right=current_portfolio, left_index=True, right_index=True, how='outer')
merge_df.columns = ['目标数量', '当前数量']




In [172]:
merge_df['trade_quantity'] = merge_df['目标数量'].sub(merge_df['当前数量'], fill_value=0)
merge_df['trade_quantity'] = (merge_df['trade_quantity']/100).round(0) *100
trade_df = merge_df[merge_df['trade_quantity'] != 0]
trade_df['price'] = trade_df.index.map(portfolio_loader._underlying_close_mapping)
trade_df['trade_amount'] = trade_df['trade_quantity'] * trade_df['price']


In [173]:
print('净买入额', trade_df['trade_amount'].sum())
print('总交易额', trade_df['trade_amount'].abs().sum())

净买入额 -10211876.0
总交易额 62432572.0


In [ ]:
# 生成交易模板
# 从index中提取代码（去掉 ' CH Equity' 后缀，取前6位）
trade_template = trade_df.copy()
trade_template['code2'] = trade_template.index.str.replace(' CH Equity', '')
trade_template['代码'] = trade_template['code2'].str[:6]

# 判断市场：根据代码前缀判断
# 根据 daily_job.py 的逻辑：如果代码对应 .XSHE（深交所），市场=1；如果对应 .XSHG（上交所），市场=0
# 0开头、3开头 -> 深交所 (.XSHE) -> 市场=1
# 6开头 -> 上交所 (.XSHG) -> 市场=0
def get_market(code):
    if pd.isna(code) or len(str(code)) < 6:
        return 0
    code_str = str(code)[:6]
    if code_str.startswith('0') or code_str.startswith('3'):
        return 1  # 深交所 (.XSHE) -> 市场=1
    elif code_str.startswith('6'):
        return 0  # 上交所 (.XSHG) -> 市场=0
    else:
        return 0  # 默认

trade_template['市场'] = trade_template['代码'].apply(get_market)

# 委托数量：取绝对值
trade_template['委托数量'] = trade_template['trade_quantity'].abs()

# 买卖方向：根据trade_quantity的正负判断，正数为买入(B)，负数为卖出(S)
trade_template['买卖方向'] = trade_template['trade_quantity'].apply(lambda x: 'B' if x > 0 else 'S')

# 其他固定字段
trade_template['产品'] = 2
trade_template['资产单元'] = 168
trade_template['投资组合'] = 173
trade_template['名称'] = None
trade_template['开平标志'] = None
trade_template['投保标志'] = None
trade_template['委托价格'] = None
trade_template['扩展业务类型'] = None
trade_template['扩展业务方向'] = None

# 选择需要的列并排序
trade_template_final = trade_template[['产品', '资产单元', '投资组合', '市场', '代码', '名称', '买卖方向', '开平标志', '投保标志', '委托价格', '委托数量',]]

# 输出交易模板
output_path = f'./Output/{date.replace(".", "")}/策略目标仓位/{main_contracts[0]}换仓交易指令模板.xlsx'
import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)
trade_template_final.to_excel(output_path, index=False)

print(f'交易模板已生成: {output_path}')
print(f'总交易数量: {len(trade_template_final)}')
print(f'买入数量: {len(trade_template_final[trade_template_final["买卖方向"] == "B"])}')
print(f'卖出数量: {len(trade_template_final[trade_template_final["买卖方向"] == "S"])}')
print(f'\n交易模板预览:')
trade_template_final.head(10)

交易模板已生成: ./Output/20260126/策略目标仓位/ZZ13_LS_SWAP001_L_01换仓交易指令模板.xlsx
总交易数量: 969
买入数量: 512
卖出数量: 457

交易模板预览:


,产品,资产单元,投资组合,市场,代码,名称,买卖方向,开平标志,投保标志,委托价格,委托数量
标的代码,,,,,,,,,,,
000012 CH Equity,2,168,173,1,000012,None,B,None,None,None,8400.0
000019 CH Equity,2,168,173,1,000019,None,B,None,None,None,1400.0
000028 CH Equity,2,168,173,1,000028,None,S,None,None,None,100.0
000029 CH Equity,2,168,173,1,000029,None,S,None,None,None,200.0
000030 CH Equity,2,168,173,1,000030,None,S,None,None,None,1700.0
000035 CH Equity,2,168,173,1,000035,None,B,None,None,None,5000.0
000048 CH Equity,2,168,173,1,000048,None,B,None,None,None,1800.0
000049 CH Equity,2,168,173,1,000049,None,B,None,None,None,200.0
000058 CH Equity,2,168,173,1,000058,None,B,None,None,None,3600.0
